In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:
# Import libraries here

from glob import glob
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from category_encoders import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils.validation import check_is_fitted


In [ ]:
# Build your `wrangle` function
def wrangle(filepath):
    
    # Read CSV file
    df = pd.read_csv(filepath)
    
    # Subset data: Apartments in "Capital Federal", less than 400,000
    mask_city = df["place_with_parent_names"].str.contains("Distrito Federal")
    mask_apt = df["property_type"] == "apartment"
    mask_price = df["price_aprox_usd"] < 100_000
    df = df[mask_city & mask_apt & mask_price]

    # Remove outliers for "surface_covered_in_m2"
    low, high = df["surface_covered_in_m2"].quantile([0.1, 0.9])
    mask_area = df["surface_covered_in_m2"].between(low,high)
    df = df[mask_area]

    # Split lat-lon to Create seperate 2 columns 
    df[["lat","lon"]] = df["lat-lon"].str.split(",",expand=True).astype(float)
    df.drop(columns="lat-lon",inplace=True)

    # Create new feature "borough" from "place_with_parent_names" column
    df["borough"] = df["place_with_parent_names"].str.split("|",expand=True)[1]
    df.drop(columns="place_with_parent_names", inplace=True)

    # Drop columns more than 50% null values
    df.drop(columns=["floor","expenses"], inplace=True)

    # Drop columns containing low- or high-cardinality categorical values
    df.drop(columns=["operation","property_type","currency","properati_url"], inplace=True)
    
    # Drop columns would constitute leakage
    df.drop(columns=[
        'price',
        'price_aprox_local_currency',
        'price_per_m2',
        'price_usd_per_m2'], inplace=True
           )
            
    # Drop columns that would create issues of multicollinearity
    df.drop(columns=["surface_total_in_m2","rooms"], inplace=True)

    return df


In [ ]:
# Use this cell to test your wrangle function on the file `mexico-city-real-estate-1.csv`
frame1 = wrangle("data/mexico-city-real-estate-1.csv")
frame1.head()


In [ ]:
files = glob("data/mexico-city-real-estate-*.csv")
files


In [ ]:
df = pd.concat([wrangle(file) for file in files], ignore_index=True)
print(df.info())
df.head()


In [ ]:
fig, ax = plt.subplots() 

# Plot the histogram on the axes object
ax.hist(df["price_aprox_usd"]) 

# Label axes using the axes 
ax.set_xlabel("Price [$]")
ax.set_ylabel("Count")

# Add title 
ax.set_title("Distribution of Apartment Prices")


In [ ]:
fig, ax = plt.subplots() 

# Create the scatter plot on the axes object
ax.scatter(x=df["surface_covered_in_m2"], y=df["price_aprox_usd"]) 

# Label axes 
ax.set_xlabel("Area [sq meters]")
ax.set_ylabel("Price [USD]")

#  Add title 
ax.set_title("Mexico City: Price vs. Area")


In [ ]:
# Plot Mapbox location and price
import plotly.express as px
fig = px.scatter_mapbox(
    df,
    lat= "lat",
    lon= "lon",
    width= 600,
    height= 600,
    color= "price_aprox_usd",
    hover_data=["price_aprox_usd"]
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()


In [ ]:
# Split data into feature matrix `X_train` and target vector `y_train`.
feature = ["surface_covered_in_m2","lat","lon","borough"]
target = "price_aprox_usd"
X_train = df[feature]
y_train = df[target]
print(X_train.shape,y_train.shape)


In [ ]:
y_mean = y_train.mean()
y_pred_baseline = [y_mean] * len(y_train)
baseline_mae = mean_absolute_error(y_train, y_pred_baseline).round(2)
print("Mean apt price:", y_mean)
print("Baseline MAE:", baseline_mae)


In [ ]:
# Build Model
model = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    SimpleImputer(),
    Ridge()
)
# Fit model
model.fit(X_train,y_train)


In [ ]:
X_test = pd.read_csv("data/mexico-city-test-features.csv")[feature]
print(X_test.info())
X_test.head()


In [ ]:
y_test_pred = pd.Series(model.predict(X_test))
y_test_pred.head()


In [ ]:
coefficients = model.named_steps["ridge"].coef_
features = model.named_steps["onehotencoder"].get_feature_names()
feat_imp = (pd.Series(model.named_steps["ridge"].coef_,
                     index=model.named_steps["onehotencoder"].get_feature_names())
           .sort_values(key=abs))
feat_imp


In [ ]:
fig, ax = plt.subplots()

feat_imp.sort_values(key=abs).tail(10).plot(kind="barh")

#  Label axes 
ax.set_xlabel("Importance [USD]") 
ax.set_ylabel("Feature")

# Add title 
ax.set_title("Feature Importances for Apartment Price")
